In [ ]:
import os
import asyncio
import aiohttp
import pandas as pd
from tqdm import tqdm
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from dotenv import load_dotenv
load_dotenv()
os.makedirs('../data/raw', exist_ok=True)

### Request data using API

In [ ]:
URL = "https://data.nayaone.com/banksim"
BATCH_SIZE = 10  # API rate limit per request
LIMIT_SIZE = 10000  # max rows (no limit = None)
OUTPUT_PATH = '../data/raw/banksim.csv'
HEADERS = {
    'Accept-Profile': 'api',
    'sandpit-key': os.getenv('API_KEY')
}

Async parallel fetch function

In [ ]:
async def fetch_page(session, offset):
    async with session.get(URL, headers=HEADERS, params={'offset': offset}) as resp:
        return offset, await resp.json()

Main parallel fetch loop

In [ ]:
CONCURRENCY = 50

In [ ]:
async def fetch_all():
    results = {}
    pbar = tqdm(total=LIMIT_SIZE, unit=" rows", desc="Fetching Data")

    connector = aiohttp.TCPConnector(
        limit=CONCURRENCY,
        ttl_dns_cache=6000,
        enable_cleanup_closed=True
    )

    sem = asyncio.Semaphore(CONCURRENCY)

    async def fetch_limited(offset):
        async with sem:
            return await fetch_page(session, offset)

    async with aiohttp.ClientSession(connector=connector) as session:
        offsets = range(0, LIMIT_SIZE, BATCH_SIZE)
        tasks = [asyncio.create_task(fetch_limited(o)) for o in offsets]

        for fut in asyncio.as_completed(tasks):
            off, data = await fut
            if not data:
                break

            results[off] = data
            pbar.update(len(data))

    pbar.close()
    return results

In [ ]:
# Run async fetcher
results = asyncio.run(fetch_all())

In [ ]:
# Flatten results in correct order
ordered_offsets = sorted(results.keys())
all_rows = [row for off in ordered_offsets for row in results[off]]

df = pd.DataFrame(all_rows)
df.to_csv(OUTPUT_PATH, index=False)

print(f"{len(all_rows)} rows saved to {OUTPUT_PATH}")